In [ ]:
import pandas as pd
import numpy as np 
import os
from pathlib import Path
import seaborn as sns

# Plotting
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.collections import PolyCollection
import matplotlib.patches as mpatches

## Prepare the data for plotting

### 1) Select same combination of algorithms and model runs for no tipping scenario to make scenarios comparable

In [ ]:
### Local paths

working_dir = Path.cwd().resolve()
repository_root = next(
    parent
    for parent in (working_dir, *working_dir.parents)
    if (parent / "07_postprocess_data" / "local" / "sdm_area_change" / "final_amazon_area_ETresid_updateds.csv").is_file()
)

# Large all-species SDM area-change table supplied in the local reproducibility-data folder
area_csv = pd.read_csv(
    repository_root
    / "07_postprocess_data"
    / "local"
    / "sdm_area_change"
    / "final_amazon_area_ETresid_updateds.csv"
)

# Small species metadata table bundled with step 04
full_species_list_amazonas = pd.read_csv(
    repository_root
    / "04_rasterize_species"
    / "local"
    / "data"
    / "full_species_list_amazon_updated.csv"
)

# Local Figure 3 output folder
save_dir = repository_root / "08_figures" / "main_figures" / "figure_3" / "output"
save_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
### Select scenario

# SSP
ssp = "ssp245"

# Dispersal scenario 
dispersal_scenario = "full_dispersal"

In [ ]:
# Prepare data
sample_structure = (
    area_csv[area_csv["tip"] == "tip"]
    [["algo", "model_run", "prec_sample"]]
    .drop_duplicates()
)

# Select no tipping data
notip = area_csv[area_csv["tip"] == "notip"].copy()

# Inner merge to add prec samples to no tipping scenario
expanded_notip = notip.merge(
    sample_structure,
    on=["algo", "model_run"],
    how="inner"   
)

# Combine
area_csv = pd.concat([
    expanded_notip,
    area_csv[area_csv["tip"] == "tip"]
], ignore_index=True)

# Clean 
area_csv["prec_sample"] = area_csv["prec_sample"].fillna(area_csv["prec_sample_y"])
area_csv = area_csv.drop(columns=['prec_sample_x',"prec_sample_y"])

### 2) Calculate median of relative change in area in amazon basin across all species

In [ ]:
median_df = (area_csv.groupby(
    ['ssp', 'tip', 'deforestation', 'time_period', "algo", "model_run", 'prec_sample'], dropna=False)
    [f'relative_change_amazon_area_{dispersal_scenario}'] 
    .median()
    .reset_index()
)

# Summarize CI statistics
summary = (
    median_df
    .groupby(['ssp', 'tip', 'deforestation', 'time_period'])
    [f'relative_change_amazon_area_{dispersal_scenario}']
    .agg([
        ('median', 'median'),
        ('lower', lambda x: np.quantile(x, 0.025)),
        ('upper', lambda x: np.quantile(x, 0.975))
    ])
    .reset_index()
)

# Exclude the no deforestation, tipping scenario
subset_df = median_df[~((median_df["tip"] == "tip") & (median_df["deforestation"] == "no_deforestation"))]

# Select ssp
subset_df = subset_df[(subset_df["ssp"] == ssp)]

In [23]:
# Use all non-bird metadata and resident-bird metadata only
# The SDMs were run for resident birds, not breeding or non-breeding birds
model_species_metadata = full_species_list_amazonas.loc[
    (full_species_list_amazonas["type"] != "birds")
    | (full_species_list_amazonas["seasonality"] == "resident")
].copy()

n_duplicate_species = model_species_metadata["Species"].duplicated().sum()
print(
    f"Model-species metadata after resident-bird selection: "
    f"{len(model_species_metadata)} rows; {n_duplicate_species} duplicate species names"
)
assert n_duplicate_species == 0, "Species metadata must contain one row per Species before merging."


Model-species metadata after resident-bird selection: 1920 rows; 0 duplicate species names


In [24]:
# Add taxonomic metadata to the SDM area results
# validate="many_to_one" ensures that each model-result row receives at most one metadata row
full_area_csv = area_csv.merge(
    model_species_metadata,
    how="left",
    left_on="species_name",
    right_on="Species",
    validate="many_to_one",
)
full_area_csv = full_area_csv.drop(columns=["Species", "PresencePoints", "area_km2"])


### 3) Calculate median of relative change in area in amazon basin by species taxa

In [ ]:
# Calculate median accross all species 
median_df_type = (full_area_csv.groupby(
    ['ssp', 'tip', 'deforestation', 'time_period', "algo", "model_run", 'prec_sample', "type"], dropna=False)
    [f'relative_change_amazon_area_{dispersal_scenario}'] 
    .median()
    .reset_index()
)

# Exclude the no deforestation, tipping scenario
subset_df_type = median_df_type[~((median_df_type["tip"] == "tip") & (median_df_type["deforestation"] == "no_deforestation"))]

# Select ssp
subset_df_type = subset_df_type[(subset_df_type["ssp"] == ssp)]

#subset_df_type["time_period"] = (
#    subset_df_type["time_period"]
#    .str.replace("_", "-", regex=False)
#)


In [ ]:
subset_df_all = subset_df.copy()
subset_df_all["type"] = "all"

subset_df_birds = subset_df_type[subset_df_type["type"] == "birds"]
subset_df_amphibians = subset_df_type[subset_df_type["type"] == "amphibians"]
subset_df_mammals = subset_df_type[subset_df_type["type"] == "mammals"]
subset_df_reptiles = subset_df_type[subset_df_type["type"] == "reptiles"]

plot_df = pd.concat([
    subset_df_all,
    subset_df_birds,
    subset_df_mammals,
    subset_df_reptiles,
    subset_df_amphibians
])


In [ ]:
type_order = ["all", "birds", "mammals", "reptiles", "amphibians"]
tip_order = ["notip", "tip"]

plot_df["group"] = (
    plot_df["tip"].astype(str) + "_" + plot_df["type"].astype(str)
)

group_order = [
    f"{tip}_{typ}"
    for tip in tip_order
    for typ in type_order
]

time_order = sorted(plot_df["time_period"].unique())

type_colors = {
    "all": "black",
    "birds": "#0262a7",
    "mammals": "#056805",
    "reptiles": "#bf5e09",
    "amphibians": "#d70400",
}

In [ ]:
# SETTINGS

tp_pretty = ["2030-2044", "2050-2069", "2080-2099"]
tp_raw = ["2030_2044", "2050_2069", "2080_2099"]
tp_map = dict(zip(tp_raw, tp_pretty))

category_order = [
    "gain",
    "small loss",
    "medium loss",
    "high loss",
    "extreme to total loss"
]

type_order = ["all", "birds", "mammals", "reptiles", "amphibians"]
tip_order = ["notip", "tip"]

plot_df["group"] = plot_df["tip"].astype(str) + "_" + plot_df["type"].astype(str)

group_order = [
    f"{tip}_{typ}"
    for tip in tip_order
    for typ in type_order
]

time_order = sorted(plot_df["time_period"].unique())

type_colors = {
    "all": "black",
    "birds": "#0e007a",
    "mammals": "#068F1ACE",
    "reptiles": "#ac025a9f",
    "amphibians": "#ff0c0794",
}


# FIGURE LAYOUT
fig = plt.figure(figsize=(18, 20))
gs = fig.add_gridspec(nrows=3, ncols=1, height_ratios=[1, 1.3, 1], hspace=0.4)

ax_violin = fig.add_subplot(gs[0, 0])

gs_bars = gs[1].subgridspec(3, 1, hspace=0.4)
axs_bars = [fig.add_subplot(gs_bars[i]) for i in range(3)]

gs_boxplots = gs[2].subgridspec(1, 3)
axs_boxplots = [fig.add_subplot(gs_boxplots[i]) for i in range(3)]

ax_box_row_title = fig.add_subplot(gs[2, 0])
ax_box_row_title.axis("off")
#ax_box_row_title.set_title(
#    "Relative Change in Loss Categories",
#    fontsize=16,
#    fontweight="bold",
#    pad=30
#)


# 1. VIOLIN PLOT
sns.violinplot(
    data=plot_df,
    x="time_period",
    y="relative_change_amazon_area_full_dispersal",
    hue="group",
    order=time_order,
    hue_order=group_order,
    dodge=True,
    cut=0,
    inner=None,
    ax=ax_violin
)

def get_type(group):
    return group.split("_", 1)[1]

# AXES STYLE 
ax_violin.tick_params(axis="both", labelsize=14)
ax_violin.set_xticklabels([tp_map[t] for t in time_order])
plt.setp(ax_violin.get_xticklabels(), fontweight="bold", fontsize=14)
plt.setp(ax_violin.get_yticklabels(), fontweight="bold", fontsize=14)

ax_violin.set_ylim(0, -100)
ax_violin.invert_yaxis()

ax_violin.set_ylabel(
    "Median change in suitable area (%)",
    fontweight="bold",
    fontsize=15,
    labelpad=10
)

ax_violin.set_xlabel("")

ax_violin.text(
    -0.1, 1.1, "a",
    transform=ax_violin.transAxes,
    fontsize=22,
    fontweight="bold",
    va="top"
)


ax_violin.grid(False)

# STYLE VIOLINS
violin_bodies = [c for c in ax_violin.collections if isinstance(c, PolyCollection)]
n_groups = len(group_order)

for i, artist in enumerate(violin_bodies):
    group = group_order[i % n_groups]
    typ = get_type(group)
    color = type_colors[typ]

    if group.startswith("notip"):
        artist.set_facecolor("none")
        artist.set_edgecolor(color)
        artist.set_linewidth(2.8)
    else:
        artist.set_facecolor(color)
        artist.set_edgecolor(color)
        artist.set_alpha(0.55)
        artist.set_linewidth(2.8)


gap = 0.1  # controls spacing 

for i, artist in enumerate(violin_bodies):
    group = group_order[i % n_groups]

    # shift only tipping scenario
    if group.startswith("tip"):
        for path in artist.get_paths():
            path.vertices[:, 0] += gap


# MEDIAN POINTS
median_df = (
    plot_df
    .groupby(["time_period", "tip", "type"])["relative_change_amazon_area_full_dispersal"]
    .median()
    .reset_index()
)

violin_positions = []
for c in violin_bodies:
    offsets = c.get_paths()[0].vertices[:, 0]
    violin_positions.append(np.mean(offsets))

group_labels = [(t, g) for t in time_order for g in group_order]
x_map = dict(zip(group_labels, violin_positions))

for _, row in median_df.iterrows():
    group = f"{row['tip']}_{row['type']}"
    x = x_map[(row["time_period"], group)]
    y = row["relative_change_amazon_area_full_dispersal"]
    color = type_colors[row["type"]]

    ax_violin.scatter(
        x, y,
        color=color,
        s=55,
        zorder=15,
        edgecolor="white",
        linewidth=1.2
    )


# LEGENDS 
ax_violin.legend_.remove()

scenario_handles = [
    mpatches.Patch(facecolor="none", edgecolor="black", label="No Tipping"),
    mpatches.Patch(facecolor="black", edgecolor="black", label="Tipping")
]

taxa_handles = []

for taxa, color in type_colors.items():
    label = taxa.capitalize()

    if taxa != "all":
        label = "                 " + label
    
    taxa_handles.append(mpatches.Patch(facecolor=color, edgecolor=color, label=label))


# LEGEND 1: Scenario 
legend1 = ax_violin.legend(
    handles=scenario_handles,
    title="Scenario",
    loc="upper center",
    bbox_to_anchor=(0.5, 1.25),
    ncol=2,
    frameon=False,
    fontsize=14
)

legend1.get_title().set_fontweight("bold")
legend1.get_title().set_fontsize(16)


ax_violin.add_artist(legend1)

# LEGEND 2: Taxa 
legend2 = ax_violin.legend(
    handles=taxa_handles,
    title="Taxa",
    loc="upper center",
    bbox_to_anchor=(0.45, -0.15),
    ncol=5,
    frameon=False,
    fontsize=14
)

legend2.get_title().set_fontweight("bold")
legend2.get_title().set_fontsize(16)

# VERTICAL LINES
xticks = ax_violin.get_xticks()

for i in range(len(xticks) - 1):
    midpoint = (xticks[i] + xticks[i + 1]) / 2
    
    ax_violin.axvline(
        x=midpoint,
        linestyle=":",
        color="gray",
        linewidth=3,
        alpha=0.7,
        zorder=0
    )

# 2. BAR CHARTS 
def classify_loss(x):
    if -100 <= x < -90: return "extreme to total loss"
    if -90 <= x < -50:  return "high loss"
    if -50 <= x < -25:  return "medium loss"
    if -25 <= x < 0:    return "small loss"
    if x >= 0:          return "gain"
    return "undefined"

mean_area = (
    area_csv
    .groupby(["species_name", "ssp", "tip", "deforestation", "time_period"])
    [f"relative_change_amazon_area_{dispersal_scenario}"]
    .mean().reset_index()
)

scenarios = [
    ("no tip", "notip", "no_deforestation"),
    ("tip + no defor", "tip", "no_deforestation"),
    ("tip + defor", "tip", "deforestation")
]

rows = []
for label, tip, defor in scenarios:
    for tp in tp_raw:
        df_s = mean_area[
            (mean_area["ssp"] == ssp) &
            (mean_area["tip"] == tip) &
            (mean_area["deforestation"] == defor) &
            (mean_area["time_period"] == tp)
        ].copy()

        df_s["loss_category"] = df_s[f"relative_change_amazon_area_{dispersal_scenario}"].apply(classify_loss)

        total = len(df_s)
        ratios = df_s["loss_category"].value_counts().reindex(category_order, fill_value=0) / total

        row = {"scenario": label, "time_period": tp}
        row.update(ratios.to_dict())
        rows.append(row)

ratio_df = pd.DataFrame(rows)

baseline = ratio_df[ratio_df["scenario"] == "no tip"]
tip_defor = ratio_df[ratio_df["scenario"] == "tip + defor"]

delta_ratio_df = baseline.merge(tip_defor, on="time_period", suffixes=("_b", "_t"))

for cat in category_order:
    delta_ratio_df[cat] = (
        (delta_ratio_df[f"{cat}_t"] - delta_ratio_df[f"{cat}_b"])
        / delta_ratio_df[f"{cat}_b"]
    ) * 100


tp_pretty_map = dict(zip(tp_raw, tp_pretty))

cmap = cm.get_cmap("PiYG")
category_colors = [cmap(v) for v in [0.85, 0.38, 0.25, 0.15, 0.01]]

for ax, tp in zip(axs_bars, tp_raw):

    df_tp = ratio_df[
        (ratio_df["scenario"].isin(["no tip", "tip + defor"])) &
        (ratio_df["time_period"] == tp)
    ]

    #  ordering
    data = np.array([
        df_tp[df_tp["scenario"] == s][category_order].values[0]
        for s in ["no tip", "tip + defor"]
    ])

    data_cum = data.cumsum(axis=1)

    ax.invert_yaxis()
    ax.get_xaxis().set_visible(False)

    y_positions = [0, 0.6]

    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        ["No Tipping", "Tipping"],
        fontsize=13,
        fontweight="bold"
    )

    for i, (cat, color) in enumerate(zip(category_order, category_colors)):
        widths = data[:, i]
        starts = data_cum[:, i] - widths

        rects = ax.barh(
            y_positions,
            widths,
            left=starts,
            height=0.35,
            color=color,
            edgecolor="black"
        )

        # text contrast
        r, g, b, _ = color
        text_color = "white" if r * g * b < 0.5 else "black"

        ax.bar_label(
            rects,
            labels=[f"{w:.2f}" if w >= 0.04 else "" for w in widths],
            label_type="center",
            fontsize=11,
            fontweight="bold",
            color=text_color
        )

    ax.set_ylabel(
        tp_pretty_map[tp],
        fontweight="bold",
        fontsize=15
    )

    ax.set_xlim(0, 1)

# clean axes
for ax in axs_bars:
    ax.set_xticks([])
    ax.set_xlabel("")

# Labels
#axs_bars[0].set_title(
#    "Proportion of Species per Loss Category",
#    fontsize=15,
#    fontweight="bold",
#    pad=12
#)

axs_bars[0].text(
    -0.08, 1.15, "b",
    transform=axs_bars[0].transAxes,
    fontsize=22,
    fontweight="bold",
    va="top"
)

# LEGEND

legend_labels = [
    "Gain (≥0%)",
    "Small Loss (-25 to 0%)",
    "Medium Loss (-50 to -25%)",
    "High Loss (-90 to -50%)",
    "Extreme to Total Loss (-100 to -90%)"
]

leg = axs_bars[-1].legend(
    handles=[plt.Rectangle((0, 0), 1, 1, color=color) for color in category_colors],
    labels=legend_labels,
    title="Loss categories",
    ncol=5,
    bbox_to_anchor=(-0.075, -0.9),
    loc="lower left",
    frameon=False,
    fontsize=14
)

leg.get_title().set_fontsize(16)
leg.get_title().set_fontweight("bold")

letters = ["c", "d", "e"]

# 3. BOX PLOTS 
for ax, tp, letter in zip(axs_boxplots, tp_raw, letters):
    row = delta_ratio_df[delta_ratio_df["time_period"] == tp]
    vals = row[category_order].iloc[0]

    ax.bar(category_order, vals, color=category_colors, edgecolor="black")
    ax.axhline(0, color="black", lw=0.8)

    ax.set_title(tp.replace("_", "–"), fontweight="bold", fontsize=16)

    ax.set_xticklabels(
        category_order,
        rotation=35,
        ha="right",
        fontweight="bold",
        fontsize=15
    )

    if tp == tp_raw[0]:
        ax.set_ylabel("Percentage change (%)", fontweight="bold", fontsize=15)
    
    ax.grid(False)
    ax.text(
        -0.08, 1.1, letter,
        transform=ax.transAxes,
        fontsize=22,
        fontweight="bold",
        va="top"
    )

plt.savefig(
    save_dir / "final_figure_3.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()